# Practice — Регуляризация Ridge / Lasso / ElasticNet (Diabetes)

10 упражнений на датасете **Diabetes** (предсказание прогрессии диабета через год). Идём: overfit → Ridge → Lasso → ElasticNet → CV-варианты → сравнение → GridSearchCV → бизнес-выбор.

Если застрял — посмотри `03_regularization_solution.ipynb`.


## О датасете

Используем **Diabetes** из `sklearn.datasets.load_diabetes()` — 442 пациента с диабетом. Каждая строка — пациент; цель — мера прогрессии заболевания через год после baseline-измерений (большее число = более тяжёлое течение).

| Колонка | Тип | Описание |
|---|---|---|
| `age` | float | Возраст (центрированный) |
| `sex` | float | Пол (бинарный признак, центрированный) |
| `bmi` | float | Body Mass Index (центрированный) |
| `bp` | float | Среднее артериальное давление |
| `s1`–`s6` | float | Шесть лабораторных биомаркеров крови |
| `target` | float | **Целевая** — прогрессия диабета через год |

Размер: 442 строки × 11 колонок.

Все признаки уже **centered и scaled** — стандартизация в Pipeline всё равно полезна для polynomial-фичей.

Бизнес-задача: модель prognosis для эндокринолога — какие признаки сильнее всего связаны с ухудшением через год. Lasso тут особенно ценен, потому что зануляет неважные биомаркеры.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV,
)
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
data = load_diabetes(as_frame=True)
df = data.frame
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'train: {X_train.shape}, test: {X_test.shape}')


## Упражнение 1 — Воспроизведи overfit

**Что делаем:** собери Pipeline `[StandardScaler, PolynomialFeatures(degree=4, include_bias=False), LinearRegression]`. Обучи на train. Посчитай RMSE отдельно на train и на test — увидишь огромный gap.

**Вход:** `X_train, y_train, X_test, y_test`.

**Ожидаемый выход:** train RMSE маленький, test RMSE большой.


In [ ]:
# твой код


## Упражнение 2 — Ridge с alpha=1

**Что делаем:** возьми тот же Pipeline, но замени `LinearRegression()` на `Ridge(alpha=1.0)`. Снова посчитай RMSE на train и test. Gap должен сильно уменьшиться.

**Вход:** `X_train, y_train, X_test, y_test`.

**Ожидаемый выход:** два числа.


In [ ]:
# твой код


## Упражнение 3 — Перебор alpha вручную

**Что делаем:** для `alpha in [0.001, 0.01, 0.1, 1, 10, 100]` обучи Ridge-Pipeline (degree=4) и выведи test RMSE. Найди оптимальный alpha (минимум RMSE).

**Вход:** train/test.

**Ожидаемый выход:** 6 строк + строка с лучшим alpha.


In [ ]:
# твой код


## Упражнение 4 — RidgeCV

**Что делаем:** собери Pipeline `[StandardScaler, PolynomialFeatures(4), RidgeCV(alphas=np.logspace(-3, 2, 30))]`. Обучи на train. Выведи `pipe.named_steps['ridge'].alpha_` и test RMSE.

**Вход:** train/test.

**Ожидаемый выход:** один alpha и одно RMSE.


In [ ]:
# твой код


## Упражнение 5 — LassoCV

**Что делаем:** аналогично, но `LassoCV(alphas=np.logspace(-3, 2, 30), max_iter=20000)`. Дополнительно посчитай, сколько коэффициентов **зануленных** (= 0).

**Вход:** train/test.

**Ожидаемый выход:** alpha, RMSE, число занулённых коэффициентов из ~120-200 polynomial-фичей.


In [ ]:
# твой код


## Упражнение 6 — ElasticNetCV

**Что делаем:** `ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], alphas=np.logspace(-3, 2, 30), max_iter=20000)`. Выведи alpha, l1_ratio, test RMSE.

**Вход:** train/test.

**Ожидаемый выход:** три значения + RMSE.


In [ ]:
# твой код


## Упражнение 7 — Сравнение в таблице

**Что делаем:** собери `pd.DataFrame` со столбцами `model, test_rmse` для:
- baseline (без регуляризации, упр. 1)
- RidgeCV (упр. 4)
- LassoCV (упр. 5)
- ElasticNetCV (упр. 6)

Округли RMSE до 3 знаков, отсортируй.

**Вход:** результаты упр. 1, 4, 5, 6.

**Ожидаемый выход:** таблица 4×2.


In [ ]:
# твой код


## Упражнение 8 — Bar chart сравнения

**Что делаем:** построй вертикальный bar chart RMSE по моделям. Заголовок «Сравнение регуляризации на Diabetes (degree=4)». Подпиши значения над столбиками.

**Вход:** таблица из упр. 7.

**Ожидаемый выход:** график.


In [ ]:
# твой код


## Упражнение 9 — GridSearchCV: degree + alpha

**Что делаем:** через `GridSearchCV` подбери совместно `poly__degree` и `ridge__alpha` для Pipeline `[StandardScaler, PolynomialFeatures(?), Ridge(?)]`.
- degree: `[2, 3, 4]`
- alpha: `np.logspace(-3, 2, 6)`
- cv=5
- scoring=`'neg_root_mean_squared_error'`

Выведи `grid.best_params_` и test RMSE лучшей модели.

**Вход:** train/test.

**Ожидаемый выход:** dict с двумя параметрами + RMSE.


In [ ]:
# твой код


## Упражнение 10 — Бизнес-выбор

**Что делаем:** В комментарии (`# вывод: ...`) одной фразой объясни эндокринологу, какую модель и почему ты бы взял в продакшн (Ridge / Lasso / ElasticNet). Опирайся на: точность (RMSE), интерпретируемость (зануляет ли коэффициенты), стабильность.

**Вход:** результаты упр. 7-9.

**Ожидаемый выход:** строка-вывод.


In [ ]:
# вывод: ...
